# Tải TCGA-LUAD và TCGA-LUSC (STAR - Counts) qua GDC API

Notebook này gọi thẳng GDC API bằng Python nên chạy được ngay trên Colab,
không phải cài Bioconductor. Nhãn giai đoạn (`stage`) được gán tự động từ
`ajcc_pathologic_stage`, không cần chỉnh tay.

Kết quả cuối cùng, lưu thẳng vào Google Drive — **1 file H5 + 1 CSV riêng cho
mỗi project** (không gộp chung):

- `tcga_luad_expr.h5`, `tcga_lusc_expr.h5` — mỗi file có 3 key: `counts` (raw),
  `tpm`, và `pheno` (nhãn `stage`/`stage_encoded` đã số hoá sẵn)
- `tcga_luad_labels.csv`, `tcga_lusc_labels.csv` — bản CSV chỉ để xem nhanh,
  không bắt buộc dùng

**Dung lượng:** mỗi project khoảng 0.5–1 GB khi tải, nên bật GPU/RAM cao không
cần thiết nhưng hãy đảm bảo còn đủ dung lượng đĩa Colab.


## 1. Thiết lập

In [ ]:
import os, io, re, json, time, tarfile, gzip
import urllib.request
import pandas as pd
import numpy as np

PROJECTS = ["TCGA-LUAD", "TCGA-LUSC"]
OUTDIR = "/content/tcga_data"
RAWDIR = os.path.join(OUTDIR, "raw")
os.makedirs(RAWDIR, exist_ok=True)

API = "https://api.gdc.cancer.gov"
BATCH = 60          # so file tai moi lan; giam neu bi timeout

print("Thu muc luu (tam, local):", OUTDIR)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTDIR = "/content/drive/MyDrive/KLTN/data/tcga"
os.makedirs(DRIVE_OUTDIR, exist_ok=True)
print("Thu muc luu tren Drive:", DRIVE_OUTDIR)

!pip install -q tables

## 2. Hàm gọi API

`_post` là wrapper có retry, vì GDC API thỉnh thoảng trả 502/503 khi tải nặng.

In [ ]:
def _post(url, payload, timeout=600, retries=4, raw=False):
    data = json.dumps(payload).encode()
    req = urllib.request.Request(
        url, data=data, headers={"Content-Type": "application/json"})
    last = None
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.read() if raw else json.loads(r.read().decode())
        except Exception as e:
            last = e
            wait = 5 * (attempt + 1)
            print(f"    loi ({e}), thu lai sau {wait}s ...")
            time.sleep(wait)
    raise RuntimeError(f"That bai sau {retries} lan: {last}")


FIELDS = ",".join([
    "file_id", "file_name",
    "cases.submitter_id",
    "cases.samples.submitter_id",
    "cases.samples.sample_type",
    "cases.samples.is_ffpe",
    "cases.diagnoses.ajcc_pathologic_stage",
    "cases.diagnoses.ajcc_pathologic_t",
    "cases.diagnoses.ajcc_pathologic_n",
    "cases.diagnoses.ajcc_pathologic_m",
    "cases.diagnoses.days_to_last_follow_up",
    "cases.diagnoses.prior_malignancy",
    "cases.demographic.gender",
    "cases.demographic.vital_status",
    "cases.demographic.days_to_death",
])


def query_files(project):
    filters = {"op": "and", "content": [
        {"op": "in", "content": {"field": "cases.project.project_id",
                                 "value": [project]}},
        {"op": "in", "content": {"field": "data_category",
                                 "value": ["Transcriptome Profiling"]}},
        {"op": "in", "content": {"field": "data_type",
                                 "value": ["Gene Expression Quantification"]}},
        {"op": "in", "content": {"field": "analysis.workflow_type",
                                 "value": ["STAR - Counts"]}},
    ]}
    payload = {"filters": filters, "fields": FIELDS,
               "format": "JSON", "size": "20000"}
    res = _post(f"{API}/files", payload)
    return res["data"]["hits"]

## 3. Lấy danh sách file và dựng bảng metadata

In [ ]:
def flatten(hits):
    rows = []
    for h in hits:
        case = (h.get("cases") or [{}])[0]
        samp = (case.get("samples") or [{}])[0]
        diag = (case.get("diagnoses") or [{}])[0]
        demo = case.get("demographic") or {}
        rows.append({
            "file_id": h["file_id"],
            "file_name": h["file_name"],
            "case_id": case.get("submitter_id"),
            "sample_id": samp.get("submitter_id"),
            "sample_type": samp.get("sample_type"),
            "is_ffpe": samp.get("is_ffpe"),
            "stage_raw": diag.get("ajcc_pathologic_stage"),
            "T": diag.get("ajcc_pathologic_t"),
            "N": diag.get("ajcc_pathologic_n"),
            "M": diag.get("ajcc_pathologic_m"),
            "gender": demo.get("gender"),
            "vital_status": demo.get("vital_status"),
            "days_to_death": demo.get("days_to_death"),
            "days_to_last_follow_up": diag.get("days_to_last_follow_up"),
        })
    return pd.DataFrame(rows)


manifests = {}
for p in PROJECTS:
    print(f"\n=== {p} ===")
    hits = query_files(p)
    mf = flatten(hits)
    manifests[p] = mf
    print(f"  {len(mf)} file")
    print(mf["sample_type"].value_counts().to_string())

## 4. Tải file

GDC cho phép POST một danh sách `file_id` tới `/data` và nhận về một tar.gz.
Tải theo lô để tránh timeout, và bỏ qua lô đã tải xong nếu bạn chạy lại.

In [ ]:
def download_batch(file_ids, dest):
    # Kiem tra file cache co san co THAT SU la gzip hop le khong (magic bytes
    # \x1f\x8b), khong chi dua vao size>0 -- vi truong hop moi phat hien: neu
    # lo chi co dung 1 file_id, GDC /data tra ve file THO (khong nen), size>0
    # nhung khong phai gzip; neu chi kiem size>0 se bi "cache" mai file sai
    # dinh dang do va khong bao gio tai lai.
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        with open(dest, "rb") as f:
            if f.read(2) == b"\x1f\x8b":
                return dest
        print(f"    {os.path.basename(dest)} co san nhung khong phai gzip hop le -- tai lai")

    blob = _post(f"{API}/data", {"ids": file_ids}, raw=True)
    with open(dest, "wb") as f:
        f.write(blob)
    return dest


def make_batches(ids, batch_size):
    """
    Chia ids thanh cac lo ~batch_size phan tu.

    QUAN TRONG: GDC /data (POST) tra ve file THO (mot file don, khong nen)
    thay vi bundle .tar.gz neu request CHI co dung 1 file_id. Dieu nay xay ra
    khi len(ids) % batch_size == 1 (lo cuoi cung con dung 1 file) -- da gap
    thuc te voi TCGA-LUAD (601 file, BATCH=60 -> lo thu 11 chi con 1 file).
    Ket qua: download_batch "thanh cong" (tai duoc, size > 0) nhung file
    khong phai tar.gz that, lam assemble() bao loi (hoac truoc day bi hieu
    nham la "file hong" roi retry vo ich vi loi lap lai y het moi lan).

    Ham nay gop phan tu le do vao lo lien truoc, dam bao KHONG lo nao con
    dung 1 phan tu (voi dieu kien tong so file >= 2, luon dung trong thuc te).
    """
    n = len(ids)
    if n == 0:
        return []
    n_batches = (n + batch_size - 1) // batch_size
    if n_batches >= 2 and n % batch_size == 1:
        n_batches -= 1
    batches, start = [], 0
    for i in range(n_batches):
        remaining_batches = n_batches - i
        remaining_items = n - start
        size = -(-remaining_items // remaining_batches)  # ceil, chia deu cho cac lo con lai
        batches.append(ids[start:start + size])
        start += size
    return batches


for p in PROJECTS:
    mf = manifests[p]
    ids = mf["file_id"].tolist()
    pdir = os.path.join(RAWDIR, p)
    os.makedirs(pdir, exist_ok=True)
    batches = make_batches(ids, BATCH)
    print(f"\n=== {p}: {len(ids)} file, {len(batches)} lo ===")
    for i, chunk in enumerate(batches):
        dest = os.path.join(pdir, f"batch_{i:03d}.tar.gz")
        print(f"  lo {i+1}/{len(batches)} ({len(chunk)} file) ...", end=" ", flush=True)
        download_batch(chunk, dest)
        print(f"{os.path.getsize(dest)/1e6:.1f} MB")


## 5. Giải nén và ghép thành ma trận

Mỗi file STAR-Counts là TSV có dòng chú thích `#`, dòng header, rồi **4 dòng
tổng hợp** `N_unmapped`, `N_multimapping`, `N_noFeature`, `N_ambiguous` phải loại bỏ.

Các cột giá trị có sẵn: `unstranded` (raw counts), `tpm_unstranded`,
`fpkm_unstranded`, `fpkm_uq_unstranded`. Notebook lấy cả counts lẫn TPM.

In [ ]:
VALUE_COLS = ["unstranded", "tpm_unstranded"]

def read_star_counts(fh):
    df = pd.read_csv(fh, sep="\t", comment="#", low_memory=False)
    df = df[~df["gene_id"].astype(str).str.startswith("N_")]
    df = df.set_index("gene_id")
    return df


def assemble(project):
    mf = manifests[project].set_index("file_id")
    pdir = os.path.join(RAWDIR, project)
    tars = sorted(f for f in os.listdir(pdir) if f.endswith(".tar.gz"))

    mats = {v: {} for v in VALUE_COLS}
    gene_meta = None

    for t in tars:
        tpath = os.path.join(pdir, t)
        # Kiem tra gzip hop le TRUOC khi mo bang tarfile -- neu khong, bao loi
        # ro rang thay vi de tarfile nem exception kho hieu. Nguyen nhan pho
        # bien nhat: lo do chi co dung 1 file_id nen GDC tra ve file tho thay
        # vi bundle .tar.gz (xem make_batches o cell tai file, da sua de
        # tranh truong hop nay -- neu van gap, xoa file nay va chay lai cell
        # tai o tren).
        with open(tpath, "rb") as fh:
            magic = fh.read(2)
        if magic != b"\x1f\x8b":
            raise RuntimeError(
                f"{t} khong phai gzip hop le (magic bytes {magic!r}, khong phai "
                f"b'\\x1f\\x8b'). Co the la file GDC tra ve tho (lo chi co 1 "
                f"file_id) hoac phan hoi loi tu GDC. Xoa {tpath} roi chay lai "
                f"cell tai file (## 4) -- ham make_batches da duoc sua de khong "
                f"con tao lo chi 1 phan tu nua."
            )
        with tarfile.open(tpath, "r:gz") as tar:
            for m in tar.getmembers():
                if not m.isfile() or not m.name.endswith(".tsv"):
                    continue
                fid = m.name.split("/")[0]
                if fid not in mf.index:
                    continue
                df = read_star_counts(io.BytesIO(tar.extractfile(m).read()))
                if gene_meta is None:
                    gene_meta = df[["gene_name", "gene_type"]].copy()
                sid = mf.loc[fid, "sample_id"]
                for v in VALUE_COLS:
                    mats[v][sid] = df[v]

    out = {v: pd.DataFrame(mats[v]) for v in VALUE_COLS}
    return out, gene_meta


expression, gene_meta = {}, None
for p in PROJECTS:
    print(f"\n=== {p} ===")
    out, gm = assemble(p)
    gene_meta = gene_meta if gene_meta is not None else gm
    expression[p] = out
    for v in VALUE_COLS:
        print(f"  {v}: {out[v].shape[0]} gene x {out[v].shape[1]} mau")


## 6. Chuẩn hoá nhãn giai đoạn

Hàm này giống hàm trong notebook GEO. Lưu ý `` không dùng được vì `"Stage IA"`
sẽ không khớp — chi tiết đã xử lý bên dưới.

Giống `geo_lung_download.ipynb`: mô thường thật (`sample_type` chứa `Normal`)
LUÔN được gán `stage="Normal"`; mẫu là khối u nhưng không đọc được giai đoạn thì
bị **loại khỏi dataset** (không giữ lại với nhãn rỗng nữa) — và `stage_encoded`/
`stage_binary_encoded` được số hoá sẵn ở đây, dùng chung `STAGE_TO_CODE` với GEO
(cùng `STAGE_ORDER`) để nhãn có cùng ý nghĩa số giữa hai nguồn, quan trọng cho
`DATA_SOURCE="merged"`.


In [ ]:
ARABIC_TO_ROMAN = {"1": "I", "2": "II", "3": "III", "4": "IV"}
MISSING = {"", "na", "n/a", "nan", "none", "null", "unknown",
           "not available", "not reported", "---", "-", "."}

def normalize_stage(x):
    """'Stage IIIB' -> 'III'. None neu khong doc duoc."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().lower()
    if s in MISSING:
        return None
    s = re.sub(r"^.*stage\s*[:=]?\s*", "", s).strip()
    if s in MISSING:
        return None
    if re.match(r"0(?![0-9])", s):
        return "0"
    m = re.match(r"(iv|iii|ii|i)(?![ivx])", s)   # khong dung \b
    if m:
        return m.group(1).upper()
    m = re.match(r"([1-4])(?![0-9])", s)
    if m:
        return ARABIC_TO_ROMAN[m.group(1)]
    return None


def binarize(stage):
    if stage in ("I", "II"):
        return "early"
    if stage in ("III", "IV"):
        return "advanced"
    return None


# Bang so hoa nhan -- DUNG CHUNG voi geo_lung_download.ipynb (cung STAGE_ORDER)
# de stage_encoded co cung y nghia giua GEO va TCGA, quan trong cho
# DATA_SOURCE="merged" trong cac notebook fair-comparison/*.ipynb.
STAGE_ORDER = ["Normal", "I", "II", "III", "IV"]
STAGE_TO_CODE = {s: i for i, s in enumerate(STAGE_ORDER)}
BINARY_TO_CODE = {"early": 0, "advanced": 1}

phenos = {}
for p in PROJECTS:
    mf = manifests[p].copy()
    mf["stage"] = mf["stage_raw"].map(normalize_stage)

    # Giong geo_lung_download.ipynb: mo thuong that (sample_type chua
    # "Normal") LUON duoc gan "Normal", bat ke stage_raw doc duoc gi (khong
    # dung du lieu stage cua mo lanh).
    mask_normal = mf["sample_type"].astype(str).str.contains("Normal", na=False)
    mf.loc[mask_normal, "stage"] = "Normal"

    mf = mf.set_index("sample_id")

    # Loai mau la KHOI U nhung khong xac dinh duoc giai doan (stage=None,
    # khong phai Normal) -- giong geo_lung_download.ipynb, vi gan nhan mo
    # thuong cho mau ung thu la SAI, khong duoc giu lai voi stage_encoded=NaN.
    ambiguous = mf["stage"].isna()
    n_ambiguous = int(ambiguous.sum())
    if n_ambiguous:
        print(f"[{p}] loai {n_ambiguous} mau la khoi u nhung khong xac dinh duoc "
              f"giai doan: {list(mf.index[ambiguous])}")
        mf = mf.loc[~ambiguous]
        for v in VALUE_COLS:
            expression[p][v] = expression[p][v].loc[:, expression[p][v].columns.isin(mf.index)]

    mf["stage_binary"] = mf["stage"].map(binarize)  # None cho "Normal"
    mf["stage_encoded"] = mf["stage"].map(STAGE_TO_CODE).astype(int)
    mf["stage_binary_encoded"] = mf["stage_binary"].map(BINARY_TO_CODE)  # NaN cho "Normal"

    phenos[p] = mf
    print(f"\n=== {p} ===")
    print(mf.groupby("sample_type")["stage"].apply(
        lambda s: s.value_counts(dropna=False).to_dict()).to_string())


## 7. Khử trùng lặp

Một case có thể có nhiều aliquot, dẫn tới nhiều mẫu cùng bệnh nhân. Với bài toán
phân loại thì để trùng sẽ làm rò rỉ dữ liệu giữa train và test. Cell này giữ mỗi
case một mẫu u, ưu tiên mẫu không phải FFPE.

In [ ]:
def dedup(pheno, expr):
    tum = pheno[pheno["sample_type"].astype(str).str.contains("Tumor", na=False)].copy()
    tum["ffpe_rank"] = tum["is_ffpe"].fillna(False).astype(bool).astype(int)
    tum = tum.sort_values(["case_id", "ffpe_rank"])
    keep_tumor = tum[~tum.index.duplicated()]
    keep_tumor = keep_tumor[~keep_tumor["case_id"].duplicated()]

    nor = pheno[pheno["sample_type"].astype(str).str.contains("Normal", na=False)]
    nor = nor[~nor.index.duplicated()]

    keep = pd.concat([keep_tumor, nor])
    keep = keep[keep.index.isin(expr.columns)]
    return keep


for p in PROJECTS:
    e = expression[p]["unstranded"]
    before = phenos[p].shape[0]
    phenos[p] = dedup(phenos[p], e)
    ids = phenos[p].index
    for v in VALUE_COLS:
        expression[p][v] = expression[p][v].loc[:, ids]
    print(f"{p}: {before} -> {len(ids)} mau sau khi khu trung lap")

## 8. Bảng tổng hợp cuối

In [ ]:
rows = []
for p in PROJECTS:
    ph = phenos[p]
    tumor = ph[ph["sample_type"].astype(str).str.contains("Tumor", na=False)]
    c = tumor["stage"].value_counts(dropna=False).to_dict()
    rows.append({
        "project": p,
        "n_tumor": len(tumor),
        "n_normal": len(ph) - len(tumor),
        "I": c.get("I", 0), "II": c.get("II", 0),
        "III": c.get("III", 0), "IV": c.get("IV", 0),
        "thieu_nhan": int(tumor["stage"].isna().sum()),
    })

df = pd.DataFrame(rows).set_index("project")
display(df)

print("\nNhan nhi phan:")
for p in PROJECTS:
    print(p, phenos[p]["stage_binary"].value_counts(dropna=False).to_dict())

## 9. Chuyển Ensembl ID sang gene symbol (tuỳ chọn)

Index hiện tại là Ensembl ID có hậu tố phiên bản (`ENSG00000000003.15`).
Bảng `gene_meta` lấy từ chính file STAR đã có sẵn `gene_name`, không cần tải thêm.

Cell này cũng lọc chỉ giữ `protein_coding` — bước gần như bắt buộc trước khi
huấn luyện, vì lncRNA và pseudogene làm tăng nhiễu.

In [ ]:
def to_symbol(expr, gene_meta, protein_coding_only=True):
    gm = gene_meta.loc[expr.index]
    if protein_coding_only:
        keep = gm["gene_type"] == "protein_coding"
        expr, gm = expr.loc[keep], gm.loc[keep]
    e = expr.copy()
    e["__sym"] = gm["gene_name"].values
    e["__var"] = expr.var(axis=1).values
    e = e.sort_values("__var", ascending=False).drop_duplicates("__sym")
    e = e.set_index("__sym").drop(columns="__var")
    e.index.name = "symbol"
    return e.sort_index()


gene_expr = {}
for p in PROJECTS:
    gene_expr[p] = {v: to_symbol(expression[p][v], gene_meta) for v in VALUE_COLS}
    print(p, {v: gene_expr[p][v].shape for v in VALUE_COLS})

## 10. Gộp nhãn + lưu vào 2 file H5 riêng (kèm CSV để xem nhanh) lên Google Drive

Cột nhãn `stage` (+ bản số hoá `stage_encoded`, `stage_binary_encoded`) đã được
gán và lọc tự động ở bước 6, không cần chỉnh tay. **Lưu riêng 1 file H5 cho mỗi
project** (`tcga_luad_expr.h5`, `tcga_lusc_expr.h5`) thay vì gộp chung 1 file —
mỗi file có 3 key bare: `counts`, `tpm`, `pheno` (không cần tiền tố project
nữa vì mỗi file đã dành riêng cho 1 project). CSV tương ứng (`tcga_luad_labels.csv`,
`tcga_lusc_labels.csv`) vẫn được ghi thêm để xem nhanh, không bắt buộc dùng.


In [ ]:
def _tcga_project_suffix(project: str) -> str:
    """'TCGA-LUAD' -> 'luad', 'TCGA-LUSC' -> 'lusc'."""
    return project.split("-")[-1].lower()


H5_PATHS = {p: os.path.join(DRIVE_OUTDIR, f"tcga_{_tcga_project_suffix(p)}_expr.h5") for p in PROJECTS}
CSV_PATHS = {p: os.path.join(DRIVE_OUTDIR, f"tcga_{_tcga_project_suffix(p)}_labels.csv") for p in PROJECTS}

for p in PROJECTS:
    pheno = phenos[p].copy()
    pheno["project"] = p

    with pd.HDFStore(H5_PATHS[p], mode="w") as store:
        store.put("counts", gene_expr[p]["unstranded"])
        store.put("tpm", gene_expr[p]["tpm_unstranded"])
        store.put("pheno", pheno)  # nhan (stage, stage_encoded, ...) luu thang trong H5

    pheno.to_csv(CSV_PATHS[p])  # ban CSV chi de xem nhanh, khong bat buoc phai dung nua

    print(f"Da luu {p}:")
    print(" ", H5_PATHS[p], "(key: 'counts', 'tpm', 'pheno')")
    print(" ", CSV_PATHS[p], pheno.shape)
    print(" ", pheno["stage"].value_counts(dropna=False).to_dict())
    print(" ", pheno["stage_encoded"].value_counts(dropna=False).sort_index().to_dict())

print("\nMa so hoa stage:", STAGE_TO_CODE)


## 11. Phụ lục — nếu bạn vẫn muốn dùng R/TCGAbiolinks

Colab có runtime R (Runtime → Change runtime type → R), nhưng cài Bioconductor
mất 20–40 phút và hay gãy vì xung đột phiên bản. Nếu vẫn muốn thử:

```r
install.packages("BiocManager")
BiocManager::install("TCGAbiolinks", ask = FALSE, update = FALSE)

library(TCGAbiolinks)
library(SummarizedExperiment)

for (p in c("TCGA-LUAD", "TCGA-LUSC")) {
  q <- GDCquery(project = p,
                data.category = "Transcriptome Profiling",
                data.type = "Gene Expression Quantification",
                workflow.type = "STAR - Counts")
  GDCdownload(q)
  se <- GDCprepare(q)
  saveRDS(se, paste0(p, ".rds"))

  # ma tran + nhan nam chung trong mot doi tuong
  # assay(se, "unstranded"); colData(se)$ajcc_pathologic_stage
}
```

Với môi trường ổn định (máy cá nhân, RStudio) thì đây vẫn là cách gọn nhất.
Trên Colab thì notebook Python ở trên đáng tin cậy hơn.


## 12. Ngắt kết nối Colab

`tcga_expr.h5` và `tcga_labels.csv` đã được lưu vào Google Drive. Cell dưới
ngắt runtime để giải phóng tài nguyên. Chỉ chạy cell này sau khi đã xác nhận
2 file trên Drive đầy đủ.

In [ ]:
missing = [p for p in PROJECTS if not (os.path.exists(H5_PATHS[p]) and os.path.exists(CSV_PATHS[p]))]
assert not missing, \
    f"Chua thay du file tren Drive cho {missing}, kiem tra lai truoc khi ngat ket noi."
print("Xac nhan da co tren Drive:", os.listdir(DRIVE_OUTDIR))

from google.colab import runtime
runtime.unassign()